# 유사한 단어 찾기 게임

1. 사전 학습된 모델 또는 적절한 데이터셋을 찾는다.
2. 워드 임베딩 모델을 학습시킨다.
3. 단어 유사도가 0.8 이상인 A, B를 랜덤 추출한다.
4. A, B와 대응되는 C를 추출한다.
5. D를 입력 받는다.

=>
A:B = C:D 관계에 대응하는 D를 찾는 게임을 만든다.
ex) A: 산, B: 바다, C: 나무, D: 물

**<출력 예시>**

관계 [ 수긍 : 추락 = 대사관 : ? ]<br>
모델이 예측한 가장 적합한 단어: 잠입<br>
당신의 답변과 모델 예측의 유사도: 0.34<br>
아쉽네요. 더 생각해보세요.

In [1]:
import pandas as pd

splits = {'train': 'dp/train-00000-of-00001.parquet', 'validation': 'dp/validation-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/klue/klue/" + splits["train"])

In [2]:
df = df['sentence']

In [3]:
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [4]:
from konlpy.tag import Okt
from tqdm import tqdm
import re

okt = Okt()

# 기존 불용어 + 확장
ko_stopwords = [
    "은","는","이","가","을","를","과","와","들","도","부터","까지","에","나","너","그","걔","얘",
    "다","하다","되다","같다","있다",
    # 의미 없는 명사성 단어 추가
    "대해","위해","통해","정도","부분","경우","사실","때문","이번","이번에","관련"
]

preprocessed_data = []

for sentence in tqdm(df):
    sentence = re.sub(r"[a-zA-Z]", " ", sentence)   # 영문 제거
    sentence = re.sub(r"\d+", " ", sentence)        # 숫자 제거
    sentence = re.sub(r"[^가-힣\s]", " ", sentence) # 특수문자 제거

    # 품사 태깅
    morphs = okt.pos(sentence, stem=True)

    # 명사만 추출 + 불용어 제거 + 길이 2 이상
    tokens = [
        word for word, tag in morphs
        if tag in ["Noun", "ProperNoun"]
        and word not in ko_stopwords
        and len(word) > 1
    ]

    preprocessed_data.append(tokens)


100%|██████████| 10000/10000 [00:17<00:00, 580.90it/s]


In [5]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_data, # corpus
    vector_size=10000,                # 임베딩 벡터 차원
    sg=0,                             # 학습 알고리즘 (0:CBOW, 1:Skip-gram)
    window=5,                         # 주변 단어 수 (앞뒤로 n개 사용) -> 이게 왜 5로 설정했는지 다시 확인
    min_count=5                       # 최소 빈도
)

In [6]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,9990,9991,9992,9993,9994,9995,9996,9997,9998,9999
숙소,-0.000125,0.036618,0.001948,-0.022112,-0.016015,-0.003949,0.031444,0.002134,0.005683,-0.004386,...,0.054459,-0.055565,-0.000166,-0.000779,-0.021547,0.042008,0.017536,-0.051108,-0.009614,-0.013125
위치,-0.000044,0.032950,0.001741,-0.019971,-0.014494,-0.003598,0.028216,0.001768,0.005250,-0.003844,...,0.049241,-0.050253,-0.000147,-0.000654,-0.019525,0.037864,0.015969,-0.046203,-0.008695,-0.011951
호스트,-0.000208,0.030711,0.001535,-0.018764,-0.013402,-0.003346,0.026443,0.001760,0.004698,-0.003634,...,0.045390,-0.046554,-0.000016,-0.000525,-0.017891,0.035274,0.014624,-0.042894,-0.008176,-0.011096
지난,-0.000212,0.064025,0.003144,-0.038633,-0.027744,-0.006319,0.054897,0.003822,0.009763,-0.007435,...,0.094558,-0.096481,0.000038,-0.000985,-0.037162,0.072827,0.030653,-0.088488,-0.016971,-0.022638
정말,-0.000211,0.029160,0.001578,-0.017701,-0.012576,-0.003095,0.025053,0.001612,0.004508,-0.003515,...,0.043338,-0.044172,-0.000179,-0.000668,-0.017132,0.033376,0.014026,-0.040783,-0.007706,-0.010501
시간,-0.000060,0.049461,0.002435,-0.029862,-0.021544,-0.005063,0.042276,0.002986,0.007557,-0.005618,...,0.073128,-0.074794,-0.000035,-0.000901,-0.028857,0.056476,0.023575,-0.068546,-0.013201,-0.017486
사진,-0.000140,0.039394,0.001845,-0.023701,-0.017140,-0.004005,0.033714,0.002219,0.006039,-0.004548,...,0.058291,-0.059461,0.000023,-0.000648,-0.023042,0.044790,0.018954,-0.054592,-0.010332,-0.013919
여행,-0.000048,0.033180,0.001606,-0.020131,-0.014354,-0.003292,0.028351,0.001829,0.005201,-0.003833,...,0.049048,-0.049989,-0.000140,-0.000656,-0.019316,0.037968,0.015737,-0.046104,-0.008778,-0.011930
매우,-0.000072,0.030861,0.001506,-0.018776,-0.013398,-0.003346,0.026373,0.001666,0.004965,-0.003653,...,0.045916,-0.046995,-0.000244,-0.000631,-0.018207,0.035394,0.014770,-0.043094,-0.008180,-0.011073
한국,-0.000164,0.059084,0.002853,-0.035699,-0.025571,-0.005869,0.050708,0.003593,0.009131,-0.006700,...,0.087254,-0.089321,0.000020,-0.000968,-0.034406,0.067375,0.028258,-0.081916,-0.015819,-0.020785


In [9]:
# model : Word2Vec
model.wv.most_similar('남자')

[('미국', 0.9999854564666748),
 ('대한', 0.9999852180480957),
 ('지난', 0.9999851584434509),
 ('오전', 0.9999850988388062),
 ('이후', 0.9999850988388062),
 ('한국', 0.9999850392341614),
 ('정부', 0.9999850392341614),
 ('방송', 0.9999849796295166),
 ('당시', 0.9999849796295166),
 ('가운데', 0.9999849200248718)]

In [16]:
import random

kv = load_model  

def play():
    vocab = list(kv.key_to_index.keys())

    # A, B 두 단어 랜덤 선택한다.
    A, B = random.sample(vocab, 2)

    # A와 가장 유사한 단어 C 선택한다.
    try:
        C = kv.most_similar(A, topn=1)[0][0]
    except KeyError:
        print("해당 단어로는 유사도 계산 불가. 다시 실행하세요.")
        return

    # 모델 예측: A:B = C:?
    try:
        pred = kv.most_similar(positive=[B, C], negative=[A], topn=1)[0][0]
    except KeyError:
        print("관계 계산 불가. 다시 실행하세요.")
        return

    print(f"관계 [ {A} : {B} = {C} : ? ]")
    print(f"모델이 예측한 가장 적합한 단어: {pred}")

    user = input("D를 입력하세요: ").strip()
    print(f"당신이 선택한 단어: {user}")

    if user in kv and pred in kv:
        sim = kv.similarity(user, pred)
        print(f"당신의 답변과 모델 예측의 유사도: {sim:.2f}")
        if sim < 0.7:
            print("아쉽네요. 더 생각해보세요.")
    else:
        print("사전에 없는 단어라 유사도 계산 불가")

In [18]:
play()

관계 [ 부정 : 변경 = 대한 : ? ]
모델이 예측한 가장 적합한 단어: 미국
당신이 선택한 단어: 영국
당신의 답변과 모델 예측의 유사도: 1.00
